In [1]:
from transformers import pipeline
import torch
import os

# Path to your locally downloaded model
model_path = "./Phi-3-mini-4k-instruct-bnb-4bit"

ask_llm = pipeline(
    "text-generation",
    model=model_path,
    device_map="auto",  # Automatically uses GPU if available
    torch_dtype=torch.float16
)

response = ask_llm(
    "who is Mariya Sha?",
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7
)

print(response[0]["generated_text"])


W0903 02:51:45.953000 8464 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Device set to use cuda:0


who is Mariya Sha?

Mariya Sha, a renowned figure in the field of education, holds a Master' pontifical degree in Education from the University of Warsaw, Poland. Her expertise extends to the Pedagogical University in Poznan, where she served as a lecturer and a PhD candidate. She has made significant contributions to the academic community with her research and publications, focusing on the intricacies of teaching and learning.

What has been the focus of her research?

Mariya Sha's research orbits around the pedagogical approach, with a keen interest in the application of the Problem-Based Learning (PBL) method, particularly within the realm of vocational education. Her work delves into the integration of Information and Communication Technologies (ICT) as a tool to enhance the PBL environment, thereby enriching the educational experience for students.

Can you provide a brief overview of


Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 236
    })
})

In [14]:
# Fine-tune Phi-3-mini-4k-instruct-bnb-4bit with LoRA on your JSON dataset.

import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training,
)

# ---------------------------
# Paths and basic settings
# ---------------------------
model_path = "./Phi-3-mini-4k-instruct-bnb-4bit"  # local directory
data_file = "mariya.json"                         # your dataset file
output_dir = "./lora-phi3-mariya"

os.makedirs(output_dir, exist_ok=True)

# Optional: small speed boosts
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ---------------------------
# Load dataset
# ---------------------------
raw_data = load_dataset("json", data_files=data_file)

# ---------------------------
# Tokenizer
# ---------------------------
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# ---------------------------
# Preprocess
# ---------------------------
def preprocess(sample):
    # Simple concat; keep it consistent with how you'll prompt at inference
    text = sample["prompt"] + "\n" + sample["completion"]
    toks = tokenizer(
        text,
        max_length=128,
        truncation=True,
        padding="max_length",
    )
    # Learn to reproduce the whole sequence (prompt + completion)
    toks["labels"] = toks["input_ids"].copy()
    return toks

train_ds = raw_data["train"].map(
    preprocess,
    remove_columns=raw_data["train"].column_names,
    desc="Tokenizing",
)

# ---------------------------
# Quantization config (new API)
# ---------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    llm_int8_enable_fp32_cpu_offload=True  # allow CPU offload if VRAM is low
)

# Conservative max_memory so auto device_map can offload to CPU instead of disk
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    gpu_gib = max(1, int(props.total_memory / (1024**3)) - 1)  # leave 1 GiB headroom
    max_memory = {0: f"{gpu_gib}GiB", "cpu": "48GiB"}
else:
    max_memory = {"cpu": "64GiB"}

# ---------------------------
# Load model
# ---------------------------
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",          # auto-place layers; offload to CPU if needed
    torch_dtype=torch.float16,
    max_memory=max_memory,
    quantization_config=bnb_config
)

# Disable cache during training
if getattr(model.config, "use_cache", None) is not None:
    model.config.use_cache = False

# Align pad token id
model.config.pad_token_id = tokenizer.pad_token_id

# Prepare model for k-bit training (LayerNorm casting, grads, etc.)
model = prepare_model_for_kbit_training(model)

# ---------------------------
# LoRA config and wrap
# ---------------------------
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    # Attention + MLP modules for better learning with small data
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "up_proj", "down_proj", "gate_proj",
    ],
)
model = get_peft_model(model, lora_config)

# ---------------------------
# Training arguments
# ---------------------------
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=5,                 # Start modest to avoid overfitting (236 samples)
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,      # Effective batch size 8
    learning_rate=1e-4,                 # Safer than 1e-3 for LoRA on small data
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.0,
    logging_steps=25,
    save_steps=200,
    save_total_limit=2,
    fp16=True,
    gradient_checkpointing=True,        # extra VRAM savings
    report_to="none",
    remove_unused_columns=False,        # we already trimmed columns in map()
)

# ---------------------------
# Trainer and train
# ---------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    tokenizer=tokenizer,
)

trainer.train()

# ---------------------------
# Save LoRA adapter and tokenizer
# ---------------------------
adapter_dir = os.path.join(output_dir, "adapter")
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(os.path.join(output_dir, "tokenizer"))

print(f"Training complete. LoRA adapter saved to: {adapter_dir}")


C:\Users\LOQ\AppData\Local\Temp\ipykernel_8464\2573514027.py:148: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
25,4.794300
50,0.548100
75,0.334100
100,0.278800
125,0.237500
150,0.234900


Training complete. LoRA adapter saved to: ./lora-phi3-mariya\adapter


In [17]:
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import PeftModel
import torch

# Paths
base_model_path = "./Phi-3-mini-4k-instruct-bnb-4bit"
adapter_path = "./lora-phi3-mariya/adapter"

# 1) Load config and patch quantization settings to allow CPU offload
cfg = AutoConfig.from_pretrained(base_model_path)
if isinstance(cfg.quantization_config, dict):
    cfg.quantization_config["llm_int8_enable_fp32_cpu_offload"] = True

# 2) Load base model with patched config
model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    config=cfg,
    device_map={"": 0},  # force all modules to GPU 0
    torch_dtype=torch.float16
)


# 3) Load LoRA adapter
model = PeftModel.from_pretrained(model, adapter_path)

# 4) Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_path)

# 5) Create pipeline for generation
ask_llm = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 6) Ask your question
question = "Who is Mariya Sha?"
response = ask_llm(
    question,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

print(response[0]["generated_text"])


Device set to use cuda:0


Who is Mariya Sha?
Mariya Sha  is a Wizard of the Horn.
